# NB05 — Entrenamiento KAN head sobre embeddings (Rama A)

**Objetivo:** Entrenar una KAN sobre los embeddings de 512 dimensiones del backbone
ResNet-18 congelado. Esta es la **Rama A** del pipeline comparativo.

**Produce:**
- `reports/models/kan_head_embeddings.pt` — pesos del KAN head entrenado
- `reports/models/kan_head_norm_stats.pt` — estadísticas de normalización (μ, σ)
  necesarias para reproducir la normalización exacta en el notebook de interpretabilidad.

**Prerequisito:** `reports/models/resnet18_tuned_best.pt` (generado en NB02).

**Siguiente paso:** `07_kan_head_interpretability.ipynb`

In [1]:
# --- Configuración ---
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

import importlib.util
if not importlib.util.find_spec('kan'):
    raise RuntimeError("PyKAN no instalado. Ejecuta: pip install pykan")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from kan import KAN

import notebook_utils as nu

# Semilla y dispositivo
SEED = 42
nu.seed_everything(SEED)
DEVICE = nu.pick_device()
print(f'Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

Device: cuda | GPU: NVIDIA GeForce RTX 5060


In [2]:
# --- Rutas ---
import paths

TUNED_BACKBONE  = ROOT / 'reports/models/resnet18_tuned_best.pt'
KAN_HEAD_PATH   = ROOT / 'reports/models/kan_head_embeddings.pt'
NORM_STATS_PATH = ROOT / 'reports/models/kan_head_norm_stats.pt'

assert TUNED_BACKBONE.exists(), f'Backbone no encontrado: {TUNED_BACKBONE}'
print('Backbone OK:', TUNED_BACKBONE)

Backbone OK: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_tuned_best.pt


In [3]:
# --- Dataset y DataLoaders ---
tr_df, val_df, test_df = nu.load_manifest_splits(
    paths.MANIFEST_TRAIN, paths.MANIFEST_TEST, seed=SEED, test_size=0.2
)
tfm = nu.default_transforms(img_size=224, augment=False)

loaders = {
    'train': DataLoader(nu.MammographyDataset(tr_df,   tfm), batch_size=32, shuffle=False, num_workers=0),
    'val':   DataLoader(nu.MammographyDataset(val_df,  tfm), batch_size=32, shuffle=False, num_workers=0),
    'test':  DataLoader(nu.MammographyDataset(test_df, tfm), batch_size=32, shuffle=False, num_workers=0),
}
print('Splits — train:', len(tr_df), '| val:', len(val_df), '| test:', len(test_df))

Splits — train: 2315 | val: 549 | test: 422


In [4]:
# --- Backbone + extracción de embeddings normalizados ---
backbone = nu.build_resnet18_backbone(TUNED_BACKBONE, DEVICE)
print('Backbone cargado (frozen, eval mode)')

embeddings, (mu, sigma) = nu.extract_and_normalize_embeddings(backbone, loaders, DEVICE)

(Xtr, ytr) = embeddings['train']
(Xva, yva) = embeddings['val']
(Xte, yte) = embeddings['test']

# Guardar estadísticas de normalización para reproducibilidad en NB07
torch.save({'mu': mu, 'sigma': sigma}, NORM_STATS_PATH)
print(f'Norm stats guardadas en: {NORM_STATS_PATH}')
print(f'Embeddings — train: {Xtr.shape} | val: {Xva.shape} | test: {Xte.shape}')

Backbone cargado (frozen, eval mode)


Norm stats guardadas en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\kan_head_norm_stats.pt
Embeddings — train: torch.Size([2315, 512]) | val: torch.Size([549, 512]) | test: torch.Size([422, 512])


In [5]:
# --- KAN head: definición y entrenamiento ---
# Hiperparámetros del KAN
KAN_WIDTH  = [512, 32, 2]  # arquitectura de la red
KAN_GRID   = 5             # resolución de la spline B
KAN_K      = 3             # orden de la spline
LR         = 5e-4
MAX_EPOCHS = 60
PATIENCE   = 8

# Mover embeddings al mismo device que el KAN
Xtr_d, ytr_d = Xtr.to(DEVICE), ytr.to(DEVICE)
Xva_d, yva_d = Xva.to(DEVICE), yva.to(DEVICE)

# Pesos de clase (manejo del desbalance)
w = nu.class_weights_from_labels(ytr.tolist(), device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=w)

kan = KAN(width=KAN_WIDTH, grid=KAN_GRID, k=KAN_K, auto_save=False, seed=SEED).to(DEVICE)
opt = torch.optim.Adam(kan.parameters(), lr=LR)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)

best_auc, best_state, wait = -1.0, None, 0
history = []

for ep in range(1, MAX_EPOCHS + 1):
    kan.train()
    opt.zero_grad()
    logits = kan(Xtr_d)
    loss = criterion(logits, ytr_d)
    loss.backward()
    opt.step()

    kan.eval()
    with torch.no_grad():
        prob = torch.softmax(kan(Xva_d), dim=1)[:, 1].cpu().numpy()
    y_true = yva_d.cpu().numpy()
    auc = float(roc_auc_score(y_true, prob)) if len(set(y_true)) > 1 else float('nan')
    sch.step(auc if not np.isnan(auc) else 0.0)

    history.append({'epoch': ep, 'loss': loss.item(), 'val_auc': auc})
    print(f'E{ep:03d}  loss={loss.item():.4f}  val_auc={auc:.4f}')

    if auc > best_auc:
        best_auc, wait = auc, 0
        best_state = {k: v.cpu().clone() for k, v in kan.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'Early stop en época {ep}')
            break

kan.load_state_dict(best_state)
kan.eval()
torch.save(kan.state_dict(), KAN_HEAD_PATH)
print(f'\nMejor val_AUC: {best_auc:.4f}')
print(f'Modelo guardado: {KAN_HEAD_PATH}')

E001  loss=0.6890  val_auc=0.6296


E002  loss=0.6628  val_auc=0.6668


E003  loss=0.6378  val_auc=0.6880


E004  loss=0.6138  val_auc=0.6977


E005  loss=0.5907  val_auc=0.7045


...

E028  loss=0.2737  val_auc=0.7192


E029  loss=0.2682  val_auc=0.7190


E030  loss=0.2656  val_auc=0.7189


E031  loss=0.2631  val_auc=0.7190


E032  loss=0.2608  val_auc=0.7191
Early stop en época 32

Mejor val_AUC: 0.7198
Modelo guardado: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\kan_head_embeddings.pt


In [6]:
# --- Evaluación final en test ---
Xte_d, yte_d = Xte.to(DEVICE), yte.to(DEVICE)

kan.eval()
with torch.no_grad():
    prob_te = torch.softmax(kan(Xte_d), dim=1)[:, 1].cpu().numpy()
    pred_te = (prob_te >= 0.5).astype(int)

thr = nu.choose_threshold_by_recall(yte.numpy(), prob_te, target_recall=0.80)
pred_thr = (prob_te >= thr).astype(int)

print('=== Métricas en TEST (umbral 0.5) ===')
m = nu.safe_binary_metrics(yte.numpy(), pred_te, prob_te)
for k, v in m.items():
    print(f'  {k:25s}: {v:.4f}')

print(f'\n=== Métricas en TEST (umbral por recall={thr:.3f}) ===')
m2 = nu.safe_binary_metrics(yte.numpy(), pred_thr, prob_te)
for k, v in m2.items():
    print(f'  {k:25s}: {v:.4f}')

=== Métricas en TEST (umbral 0.5) ===
  acc                      : 0.6469
  recall_malignant         : 0.7471
  precision_malignant      : 0.5532
  f1_malignant             : 0.6357
  auc                      : 0.6932

=== Métricas en TEST (umbral por recall=0.450) ===
  acc                      : 0.6374
  recall_malignant         : 0.8103
  precision_malignant      : 0.5402
  f1_malignant             : 0.6483
  auc                      : 0.6932
